# Welding Sensors, Visually: Sensor Correlation & Frames per Temperature Measurement

This notebook contains **interactive plots** (sliders). Two sections:

| Section | Concept | Knobs you can turn |
|---|---|---|
| 1 | Thermal camera vs. current sensor as predictors of weld quality | per-sensor true ρ and noise, number of welds, confidence |
| 2 | One weld, many thermal frames – how sure is the frame-averaged temperature? | number of frames, true temperature, frame-to-frame σ, confidence |


**Two kinds of controls** (each plot has two labelled boxes):

| Box | Sliders | Meaning |
|---|---|---|
| 🌍 **Simulated world** (red) | ρ and noise of each sensor (section 1); true temperature and frame σ (section 2) | The physical truth. In real life you **never know** these – they are what you are trying to estimate. We only set them here so we can generate fake data and check whether the statistics recover the truth. |
| 🎛️ **In your hands** (green) | number of welds / frames, confidence, random seed | Choices you really make: how much data to collect, how sure you want to be, and the **random seed** (which particular random sample you happened to get – re-run it to see another "lucky or unlucky" sample). |

> Running theme: a **sample** is a small, noisy window onto a **population**. Statistics tells us how much to trust that window.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import IntSlider, FloatSlider, Dropdown

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

RNG_SEED = 42

def grouped_controls(func, world, hands):
    """Show `func` with two labelled control boxes: the simulated world vs. what you control."""
    for w in [*world.values(), *hands.values()]:
        w.style = {"description_width": "110px"}
        w.layout = widgets.Layout(width="330px")
    def box(title, subtitle, ws, color):
        head = widgets.HTML(f"<b>{title}</b><br><span style='font-size:90%;opacity:0.8'>{subtitle}</span>")
        return widgets.VBox([head, *ws.values()], layout=widgets.Layout(
            border=f"2px solid {color}", padding="8px 12px", margin="0 12px 8px 0"))
    panel = widgets.HBox([
        box("🌍 Simulated world", "physical truth – unknown in real life, we only simulate it", world, "tab:red"),
        box("🎛️ In your hands", "choices you make when you collect and analyse data", hands, "tab:green"),
    ])
    out = widgets.interactive_output(func, {**world, **hands})
    display(widgets.VBox([panel, out]))


print("Setup done. Move the sliders below each plot to explore.")

Setup done. Move the sliders below each plot to explore.


### 1. Two sensors on a welding line – which one predicts weld quality better?

Every weld gets a **quality score (0–100 %)** from destructive testing. During the weld, two sensors record
a single summary number each:

| Sensor | Reading used | Typical range |
|---|---|---|
| **Thermal camera** | mean temperature of the weld-pool image | ≈ 1200 – 1600 °C |
| **Current sensor** | mean weld current over the weld | ≈ 140 – 260 A |

Each sensor has its own **true correlation $\rho$** with weld quality (how well it *could* predict quality if it were perfectly precise) and its own **noise** level.
We weld $n$ parts, test them, and ask: *which sensor's reading tracks quality, and how sure are we?*

**What does "noise" mean here?** It is the standard deviation of random Gaussian jitter added to each **sensor reading**,
expressed in **z-score units of the signal** (the clean reading has std = 1 before being put into °C or A).
So `noise = 1.0` means a typical reading is off by about one standard deviation of the true signal;
`noise = 0.2` is a precise sensor, `noise = 2.0` is a very sloppy one.
It is *not* the z-score of one point – it is the spread of the error distribution.

Noise doesn't change the underlying physics, it **dilutes** the correlation you can observe:

$$r_{\text{expected}} = \frac{\rho}{\sqrt{1 + \text{noise}^2}}
\qquad\Rightarrow\qquad
R^2_{\text{observed}} \approx \frac{\rho^2}{1 + \text{noise}^2}$$

**Plot layout** – blue = thermal camera, orange = current sensor everywhere.
* **Top-left**: weld quality (y) against each sensor's reading. Because the two sensors have different units, the thermal camera uses the **bottom** x-axis (°C) and the current sensor the **top** x-axis (A). Lines = fits, bands = ±1 residual std.
* **Top-right**: for each sensor the sample $r$ (solid), its CI (shaded), the bootstrap distribution (histogram) and the dashed *expected* $r$ after noise dilution.
* **Bottom**: *how many welds do I need?* The x-axis is the number of welds $k$ from 4 up to the selected $n$. The y-axis is the **width of the CI for $r$** – i.e. how big the range of plausible correlations still is. Dashed = theoretical width around the expected $r$; solid = the width you'd actually have reported after the first $k$ welds of this sample; dot = final width at $n$. Dotted grey lines mark "r known to ±0.20" and "±0.10".

Below the plots: a summary table and a **Fisher z-test** asking whether the two sensors' correlations are statistically different.

> **Note on identical settings.** Each sensor draws its own independent random sample. If you give both sensors the *same* $\rho$ and noise, the two will look **similar but not identical** – same expected $r$, same CI width, but different individual welds and a slightly different observed $r$. That is the correct behaviour: two equally good sensors measuring the same thing should agree *statistically*, not point-for-point.
**Controls** – 🌍 *simulated world* (unknown in real life): thermal ρ, thermal noise, current ρ, current noise. 🎛️ *in your hands*: number of welds $n$, confidence level, random seed.


In [2]:
WELD_SENSORS = {
    "Thermal camera": dict(what="mean image temperature", unit="°C", center=1400, scale=80, color="tab:blue"),
    "Current sensor": dict(what="mean weld current",      unit="A",  center=200,  scale=25, color="tab:orange"),
}


def _simulate_weld_sensor(rho_true, noise_std, n, rng, center, scale):
    """Weld quality (0–100 %) plus a sensor reading correlated with it at ρ. Sensor noise is added to the *reading*."""
    q = rng.normal(0, 1, n)                                                        # latent weld quality, unit variance
    x_clean = rho_true * q + np.sqrt(max(1 - rho_true**2, 1e-12)) * rng.normal(0, 1, n)
    x_obs = x_clean + noise_std * rng.normal(0, 1, n)                              # noise in z-score units of the signal
    quality = np.clip(70 + 10 * q, 0, 100)
    return center + scale * x_obs, quality


def _analyse(x, y, confidence, rng, n_boot=1000):
    n = len(x)
    r, p = stats.pearsonr(x, y)
    slope, intercept, *_ = stats.linregress(x, y)
    resid_std = (y - (intercept + slope * x)).std(ddof=2)
    z_crit = stats.norm.ppf(1 - (1 - confidence) / 2)
    ci = np.tanh(np.arctanh(r) + np.array([-1, 1]) * z_crit / np.sqrt(n - 3))
    idx = rng.integers(0, n, size=(n_boot, n))
    xb, yb = x[idx], y[idx]
    xc, yc = xb - xb.mean(1, keepdims=True), yb - yb.mean(1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        boot_r = (xc * yc).sum(1) / np.sqrt((xc**2).sum(1) * (yc**2).sum(1))
    return dict(r=r, p=p, slope=slope, intercept=intercept, resid_std=resid_std,
                ci=ci, boot_r=boot_r[np.isfinite(boot_r)])


def _ci_width(r, k, z_crit):
    """Width of the Fisher CI for a correlation r estimated from k points."""
    zr = np.arctanh(np.clip(r, -0.999, 0.999))
    return np.tanh(zr + z_crit / np.sqrt(k - 3)) - np.tanh(zr - z_crit / np.sqrt(k - 3))


def _running_ci_width(x, y, ks, z_crit):
    """CI width you would have reported after only the first k readings."""
    out = np.empty(len(ks))
    for i, k in enumerate(ks):
        with np.errstate(invalid="ignore", divide="ignore"):
            out[i] = _ci_width(np.corrcoef(x[:k], y[:k])[0, 1], k, z_crit)
    return out


def compare_weld_sensors(rho_thermal=0.90, noise_thermal=0.2, rho_current=0.90, noise_current=1.5,
                         n_samples=50, confidence=0.95, seed=RNG_SEED):
    # Each sensor gets its own independent random stream so identical settings still give different draws
    sensors = [("Thermal camera", rho_thermal, noise_thermal, np.random.default_rng(seed)),
               ("Current sensor", rho_current, noise_current, np.random.default_rng(seed + 10_000))]
    alpha = 1 - confidence
    z_crit = stats.norm.ppf(1 - alpha / 2)

    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.4, 1], height_ratios=[1, 0.9])
    ax_sc, ax_r, ax_n = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
    ax_sc2 = ax_sc.twiny()                                 # second x-axis: the two sensors have different units
    ax_sc2.grid(False)
    x_axes = {"Thermal camera": ax_sc, "Current sensor": ax_sc2}

    ks = np.arange(4, n_samples + 1)                       # Fisher CI needs n ≥ 4
    results = []
    for name, rho_true, noise, rng in sensors:
        meta = WELD_SENSORS[name]
        color = meta["color"]
        x, y = _simulate_weld_sensor(rho_true, noise, n_samples, rng, meta["center"], meta["scale"])
        res = _analyse(x, y, confidence, rng)
        r_expected = rho_true / np.sqrt(1 + noise**2)      # noise dilutes ρ
        results.append((name, rho_true, noise, r_expected, res))
        sig = res["p"] < alpha

        # ---- Top-left: both sensors vs weld quality (each on its own x-axis) -----------------
        ax = x_axes[name]
        xs = np.linspace(x.min(), x.max(), 100)
        ys = res["intercept"] + res["slope"] * xs
        # ax.scatter(x, y, color=color, alpha=0.55, edgecolor="k", lw=0.3, s=28,
        #            label=f"{name} (ρ = {rho_true:+.2f}, noise = {noise:.1f} σ)  →  r = {res['r']:+.2f}, "
        #                  f"p = {res['p']:.2g} {'✓' if sig else '✗'}")
        ax.scatter(x, y, color=color, alpha=0.55, edgecolor="k", lw=0.3, s=28,
                   label=f"{name} →  r = {res['r']:+.2f}, "
                         f"p = {res['p']:.2g} {'✓' if sig else '✗'}")
        ax.plot(xs, ys, color=color, lw=2.2)
        ax.fill_between(xs, ys - res["resid_std"], ys + res["resid_std"], color=color, alpha=0.08)
        pad = 0.05 * (x.max() - x.min())
        ax.set_xlim(x.min() - pad, x.max() + pad)
        ax.set_xlabel(f"{name}: {meta['what']} ({meta['unit']})", color=color)
        ax.tick_params(axis="x", colors=color)

        # ---- Top-right: how sure are we about r? -------------------------------------------
        ax_r.hist(res["boot_r"], bins=40, color=color, alpha=0.3, density=True)
        ax_r.axvspan(*res["ci"], color=color, alpha=0.15)
        ax_r.axvline(res["r"], color=color, lw=2.5,
                     label=f"{name}: r = {res['r']:+.2f},  {confidence:.0%} CI [{res['ci'][0]:+.2f}, {res['ci'][1]:+.2f}]")
        ax_r.axvline(r_expected, color=color, ls="--", lw=1.5, label=f"{name}: expected r = {r_expected:+.2f}")

        # ---- Bottom: width of the CI for r vs number of welds measured ----------------------
        ax_n.plot(ks, _ci_width(r_expected, ks, z_crit), color=color, ls="--", lw=1.5,
                  label=f"{name}: expected CI width (around r = {r_expected:+.2f})")
        ax_n.plot(ks, _running_ci_width(x, y, ks, z_crit), color=color, lw=1.8, alpha=0.9,
                  label=f"{name}: CI width you'd have reported after k welds")
        ax_n.plot(n_samples, res["ci"][1] - res["ci"][0], "o", color=color, ms=8, mec="k")

    ax_sc.set_ylabel("Weld quality (%)")
    ax_sc.set_ylim(min(30, ax_sc.get_ylim()[0]), 105)
    ax_sc.set_title(f"Weld quality vs. sensor reading, n = {n_samples} welds each  (line = fit, band = ±1 residual std)", pad=28)
    h1, l1 = ax_sc.get_legend_handles_labels()
    h2, l2 = ax_sc2.get_legend_handles_labels()
    ax_sc.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8.5)

    ax_r.axvline(0, color="tab:red", ls=":", lw=1.5, label="r = 0 (no relation)")
    ax_r.set_xlim(-1.05, 1.05)
    ax_r.set_xlabel("Correlation coefficient r  (sensor reading ↔ weld quality)")
    ax_r.set_title("How sure are we about r?  (histogram = bootstrap, shaded = CI)")
    ax_r.legend(loc="upper left", fontsize=8)

    for w, lab in ((0.4, "±0.20"), (0.2, "±0.10")):
        ax_n.axhline(w, color="grey", ls=":", lw=1)
        ax_n.text(ks[0] + 0.5, w + 0.02, f"r known to {lab}", fontsize=8, color="grey")
    ax_n.set_xlim(ks[0], n_samples)
    ax_n.set_ylim(0, 2.0)
    ax_n.set_xlabel("Number of welds measured k  (max = selected n)")
    ax_n.set_ylabel(f"Width of the {confidence:.0%} CI for r")
    ax_n.set_title("How wide is the range of plausible r after k welds?  (dashed = theory, solid = this sample, dot = final)")
    ax_n.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

    # ---- Summary & direct comparison of the two correlations (Fisher z-test) -------------------
    print(f"{'':16s}{'true ρ':>9s}{'noise σ':>9s}{'expected r':>12s}{'observed r':>12s}{'r²':>7s}"
          f"{'CI low':>9s}{'CI high':>9s}{'CI width':>10s}{'p-value':>10s}")
    for name, rho_true, noise, r_exp, res in results:
        lo, hi = res["ci"]
        print(f"{name:16s}{rho_true:+9.2f}{noise:9.1f}{r_exp:+12.2f}{res['r']:+12.3f}{res['r']**2:7.2f}"
              f"{lo:+9.2f}{hi:+9.2f}{hi-lo:10.2f}{res['p']:10.3g}")

    r1, r2 = results[0][4]["r"], results[1][4]["r"]
    z_diff = (np.arctanh(r1) - np.arctanh(r2)) / np.sqrt(2 / (n_samples - 3))
    p_diff = 2 * (1 - stats.norm.cdf(abs(z_diff)))
    overlap = max(results[0][4]["ci"][0], results[1][4]["ci"][0]) <= min(results[0][4]["ci"][1], results[1][4]["ci"][1])
    print(f"\nDo the two {confidence:.0%} CIs overlap?            {'yes' if overlap else 'no'}")
    print(f"Fisher z-test, H₀: ρ₁ = ρ₂           z = {z_diff:+.2f},  p = {p_diff:.3g}  → "
          f"{'sensors differ' if p_diff < alpha else 'cannot say the sensors differ'} at α = {alpha:.0%}")


grouped_controls(
    compare_weld_sensors,
    world=dict(  # physical truth: unknown in real life, we only simulate it here
        rho_thermal=FloatSlider(value=0.90, min=-0.99, max=0.99, step=0.01, description="Thermal ρ"),
        noise_thermal=FloatSlider(value=0.2, min=0.0, max=3.0, step=0.1, description="Thermal noise σ"),
        rho_current=FloatSlider(value=0.90, min=-0.99, max=0.99, step=0.01, description="Current ρ"),
        noise_current=FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, description="Current noise σ"),
    ),
    hands=dict(  # what we choose when collecting / analysing data
        n_samples=IntSlider(value=50, min=5, max=500, step=5, description="Welds n"),
        confidence=Dropdown(options=[0.90, 0.95, 0.99], value=0.95, description="Confidence"),
        seed=IntSlider(value=RNG_SEED, min=0, max=100, step=1, description="Random seed"),
    ),
)


### Try this
1. **Defaults** (both ρ = 0.90, thermal noise 0.2, current noise 1.5). Same underlying physics, but the current sensor's observed $r$ drops from ≈ 0.88 to ≈ 0.50 and its CI is much wider. The noise, not the physics, is what you're seeing.
2. **Same settings for both** (e.g. ρ = 0.90, noise 0.2 on both). The two overlap heavily but not perfectly – different random welds, same statistics. The Fisher test should say "cannot say the sensors differ".
3. **n = 10.** Both CIs balloon, they overlap heavily, and the Fisher test says "cannot say the sensors differ" – even though one is clearly worse. Too few welds to rank sensors.
4. **n = 300.** CIs tighten, stop overlapping, p for the difference becomes tiny – now you have evidence the thermal camera is the better predictor. In the bottom chart, read off where each curve crosses the "±0.10" line – that's how many welds each sensor needs for a usable $r$.
5. **Thermal ρ = 0.55, noise 0; Current ρ = 0.90, noise 1.5.** Expected $r$ is about the same (≈ 0.55 vs 0.50) – a *weak-but-clean* predictor and a *strong-but-noisy* one can look identical in the data. Only knowing the noise lets you tell them apart.
6. **Negative ρ.** Set Current ρ = −0.90 (higher current → worse weld): the fit slopes down, $r$ is negative, and the dilution formula still applies symmetrically.
7. **Cycle the seed at n = 50 and watch the bottom chart.** The solid "reported width" line jumps around the dashed theory line, and is far above it when early $r$ happens to be small – the noisy sensor's line is especially erratic.

### Conclusions
* **Sensor noise directly lowers the correlation you can observe** by $1/\sqrt{1+\text{noise}^2}$. A low $r$ may mean a weak physical link *or* a noisy instrument – the scatter plot alone can't tell you which.
* **Noisy sensors need more welds** to reach the same CI width; the CI for $r$ is widest when $r$ is near 0 and narrows as $|r| \to 1$. The bottom chart lets you read the required $n$ directly.
* **Early welds are misleading.** With 5–10 welds the CI for $r$ spans most of [−1, 1]; do not pick a sensor until the width has dropped below what you can live with.
* **Compare sensors with a test, not by eye.** Two $r$ values that look different may have overlapping CIs; use the Fisher z-test (or non-overlapping CIs as a rough check) before claiming one is better.
* **Noise is in z-score units of the signal** – a convenient, scale-free way to specify instrument precision: "noise = 0.5" means the error spread is half the signal spread, whether the units are °C or A.

### 2. One weld, many images – how sure are we about *its* mean temperature?

Zoom in on a **single weld**. The thermal camera fires $n$ times during the weld and each frame gives one
mean-image temperature. Frame-to-frame the reading jitters (sensor noise, plume, reflections), so the $n$ readings
scatter around the weld's **true temperature** $\mu$ with some spread $\sigma_{\text{frame}}$.
We report the **average of the $n$ frames** as "the temperature of this weld". How much can we trust that number?

* **One frame** is a draw from $\mathcal{N}(\mu, \sigma_{\text{frame}}^2)$ – the wide curve.
* **The mean of $n$ frames** is a draw from $\mathcal{N}\!\left(\mu, \sigma_{\text{frame}}^2 / n\right)$ – the narrow curve. Its spread is the **standard error**
  $$SE = \frac{\sigma_{\text{frame}}}{\sqrt{n}}$$
* In practice we don't know $\sigma_{\text{frame}}$; we estimate it from the frames themselves ($s$) and use the t-distribution:
  $$\bar{T} \;\pm\; t_{\alpha/2,\,n-1}\cdot\frac{s}{\sqrt{n}}$$

**Plot layout**
* **Left**: the wide Gaussian of a single frame, the narrow Gaussian of the $n$-frame mean, the actual frames you took (ticks), their mean (vertical line) and its confidence interval (shaded). The true temperature is the dashed black line.
* **Right**: how the CI half-width shrinks as you take more frames – theory (solid) and what *this* run would have reported after the first $k$ frames (dots). Dotted guides at ±5 °C and ±2 °C.
**Controls** – 🌍 *simulated world* (unknown in real life): true weld temperature μ, frame-to-frame σ. 🎛️ *in your hands*: number of frames $n$, confidence level, random seed.


In [ ]:
def plot_weld_mean_temperature(n_frames=10, true_temp=1400.0, frame_std=25.0, confidence=0.95, seed=RNG_SEED):
    alpha = 1 - confidence
    rng = np.random.default_rng(seed)
    frames = rng.normal(true_temp, frame_std, n_frames)
    t_bar, s = frames.mean(), frames.std(ddof=1)
    se_true = frame_std / np.sqrt(n_frames)                      # theory, σ known
    t_crit = stats.t.ppf(1 - alpha / 2, n_frames - 1)
    half = t_crit * s / np.sqrt(n_frames)                        # practice, σ estimated from the frames
    ci_lo, ci_hi = t_bar - half, t_bar + half
    captured = ci_lo <= true_temp <= ci_hi

    fig, (ax, ax_n) = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1.5, 1]})

    # ---- Left: one-frame Gaussian vs n-frame-mean Gaussian ---------------------------------
    x_lo, x_hi = true_temp - 4 * frame_std, true_temp + 4 * frame_std
    xg = np.linspace(x_lo, x_hi, 800)
    pdf_frame = stats.norm.pdf(xg, true_temp, frame_std)
    pdf_mean = stats.norm.pdf(xg, true_temp, se_true)
    ax.plot(xg, pdf_frame, color="tab:orange", lw=2, label=f"One frame:  N(μ, {frame_std:.0f}²)")
    ax.fill_between(xg, pdf_frame, color="tab:orange", alpha=0.12)
    ax.plot(xg, pdf_mean, color="tab:blue", lw=2.5,
            label=f"Mean of {n_frames} frames:  N(μ, ({frame_std:.0f}/√{n_frames})²) → SE = {se_true:.1f} °C")
    ax.fill_between(xg, pdf_mean, color="tab:blue", alpha=0.18)

    ax.axvline(true_temp, color="black", ls="--", lw=1.5, label=f"True weld temperature μ = {true_temp:.0f} °C")
    ymax = pdf_mean.max() * 1.12
#     ax.plot(frames, np.full(n_frames, -0.02 * ymax), "|", color="tab:orange", ms=16, mew=1.5,
#             label=f"The {n_frames} frames you actually took")
    ax.axvline(t_bar, color="tab:red", lw=2.5, label=f"sample mean T̄ = {t_bar:.1f} °C")
    clipped = ci_lo < x_lo or ci_hi > x_hi                       # tiny n → CI wider than the plot
    ax.axvspan(max(ci_lo, x_lo), min(ci_hi, x_hi), color="tab:red", alpha=0.18,
               label=f"{confidence:.0%} CI for μ:  T̄ ± {half:.1f} °C  =  [{ci_lo:.1f}, {ci_hi:.1f}]  "
                     f"{'✓ contains μ' if captured else '✗ misses μ'}{'  (extends beyond plot!)' if clipped else ''}")
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(-0.05 * ymax, ymax)
    ax.set_xlabel("Mean image temperature (°C)")
    ax.set_ylabel("Probability density")
    ax.set_title(f"One weld, {n_frames} thermal frames: how well does the frame-average pin down μ?")
    ax.legend(loc="upper left", fontsize=8.5)

    # ---- Right: CI half-width vs number of frames --------------------------------------------
    ks = np.arange(2, max(n_frames, 60) + 1)
    theory = stats.t.ppf(1 - alpha / 2, ks - 1) * frame_std / np.sqrt(ks)
#     ax_n.plot(ks, theory, color="tab:blue", lw=2, label=f"Theory: t·σ/√k  (σ = {frame_std:.0f} °C)")
    kk = np.arange(2, n_frames + 1)
    reported = [stats.t.ppf(1 - alpha / 2, k - 1) * frames[:k].std(ddof=1) / np.sqrt(k) for k in kk]
    ax_n.plot(kk, reported, "o-", color="tab:red", ms=4, lw=1, alpha=0.8,
              label="What this run would have reported after k frames")
    ax_n.plot(n_frames, half, "o", color="tab:red", ms=10, mec="k", label=f"k = {n_frames}: ± {half:.1f} °C")
    y_top = min(theory[1] * 1.3, 4 * frame_std)                  # start from k = 3 so the k = 2 spike doesn't flatten the curve
    for lvl in (5, 2):
        if lvl < y_top:
            ax_n.axhline(lvl, color="grey", ls=":", lw=1)
            ax_n.text(ks[-1], lvl + 0.01 * y_top, f"± {lvl} °C", ha="right", fontsize=8, color="grey")
    ax_n.set_xlim(ks[0], ks[-1])
    ax_n.set_ylim(0, y_top)
    ax_n.set_xlabel("Number of frames averaged k")
    ax_n.set_ylabel(f"{confidence:.0%} CI half-width (°C)")
    ax_n.set_title("More frames → tighter mean  (shrinks like 1/√k)")
    ax_n.legend(loc="upper right", fontsize=8.5)
    plt.tight_layout()
    plt.show()

    n_for = lambda target: int(np.ceil((stats.norm.ppf(1 - alpha / 2) * frame_std / target) ** 2))
    print(f"Frames: n = {n_frames},  T̄ = {t_bar:.2f} °C,  s = {s:.2f} °C   (true μ = {true_temp:.0f}, σ_frame = {frame_std:.0f})")
    print(f"Standard error of the mean  : σ/√n = {se_true:.2f} °C   (estimated s/√n = {s/np.sqrt(n_frames):.2f} °C)")
    print(f"{confidence:.0%} CI for μ              : [{ci_lo:.2f}, {ci_hi:.2f}]  → ± {half:.2f} °C   {'(contains μ)' if captured else '(MISSES μ – happens ~' + f'{alpha:.0%} of runs)'}")
    print(f"Frames needed for ± 5 °C    : ≈ {n_for(5)}      for ± 2 °C: ≈ {n_for(2)}      for ± 1 °C: ≈ {n_for(1)}")


grouped_controls(
    plot_weld_mean_temperature,
    world=dict(  # physical truth: unknown in real life, we only simulate it here
        true_temp=FloatSlider(value=1400, min=1200, max=1600, step=10, description="True T (°C)"),
        frame_std=FloatSlider(value=25, min=2, max=80, step=1, description="Frame σ (°C)"),
    ),
    hands=dict(  # what we choose when collecting / analysing data
        n_frames=IntSlider(value=10, min=2, max=200, step=1, description="Frames n"),
        confidence=Dropdown(options=[0.90, 0.95, 0.99], value=0.95, description="Confidence"),
        seed=IntSlider(value=RNG_SEED, min=0, max=100, step=1, description="Random seed"),
    ),
)


### Try this
1. **n = 2 → 10 → 100.** The blue curve (distribution of the mean) gets dramatically narrower while the orange curve (one frame) does not move. The CI half-width drops from tens of degrees to a few degrees.
2. **Frame σ = 60 °C** (a hot, splattery weld). With 10 frames you only know μ to about ±40 °C. Read off the right chart how many frames you'd need for ±5 °C – it's hundreds.
3. **Cycle the seed at n = 5.** The red mean jumps around and roughly 1 in 20 runs the CI misses μ entirely – that is exactly the 5 % error rate of a 95 % CI.
4. **Confidence 0.99.** Same frames, wider interval. You can't get more certainty for free – only by taking more frames.
5. **n = 2.** The t-critical value is 12.7 (not 1.96) – with one degree of freedom you know almost nothing about σ, so the interval explodes even if the two frames happen to agree.

### Conclusions
* **Averaging frames does not make the camera more precise – it makes the *average* more precise.** One frame is still ±σ; the mean of $n$ frames is ±σ/√n.
* **Diminishing returns.** Going from 1 → 4 frames halves the uncertainty; going from 4 → 16 halves it again. Each halving costs 4× the frames.
* **Report the CI, not just the mean.** "1400 °C" from 3 frames and "1400 °C" from 100 frames are very different statements. In section 1 the per-weld temperature was treated as a single reading – its noise σ is exactly this SE, so **more frames per weld = less sensor noise = higher observable correlation with quality**.
* **Decide the precision you need first**, then set the frame count: $n \approx (z\,\sigma_{\text{frame}} / \text{target})^2$.